# 02. Context 分支的量化（硬件友好 QAT）

配套 md 教程 [05](../05-quantization-motivation.md)、[06](../06-hw-friendly-ops-qat.md)、
[07](../07-export-numpy-verify.md) 章讲了量化的动机、算子替换原理和导出验证方法。
这个 notebook 把那套方法**真正应用到 notebook 01 训练出的 Context 分支上**，
对应真实项目的 `train_qat.py --model context_hw`。

## 目标算子替换（回顾,详见教程 06 章）

| 原始算子 | 硬件友好替换 |
|---|---|
| `nn.LayerNorm` | `HWAffine`（逐通道仿射,数据无关,可折叠进前一层 Linear）|
| Linear/Conv 权重(FP32) | 逐输出通道对称 INT4 |
| 激活/膜电位(FP32) | Q8.8 定点(步长 1/256,范围 ±128) |
| 衰减系数 beta(FP32) | k/256 网格 |

Context 分支本身没有 GELU、没有除法，所以只需要替换 LayerNorm 和做权重/激活/衰减量化。
这套 INT4+QAT 的方案是三个分支里最"重"的一种量化——后面 Delay 分支会看到一种简单得多、
不需要重新训练的量化方式，两者的对比本身就是一个重要的教学点。

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("training/semg_snn_90_loop")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset
from hw_model import HWClassAdaptiveContextSNN, remap_context_state
from hw_fixed_reference import HWFixedContext, affine, linear, spike as np_spike

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NB_RUNS = PROJECT_ROOT / "runs_notebook"
NB_RUNS.mkdir(exist_ok=True)
WEIGHTS_DIR = NB_RUNS / "weights_hw"
WEIGHTS_DIR.mkdir(exist_ok=True)

FP32_CONTEXT_CHECKPOINT = NB_RUNS / "context23_stage2_stream_nb" / "best.pt"
if not FP32_CONTEXT_CHECKPOINT.exists():
    FP32_CONTEXT_CHECKPOINT = PROJECT_ROOT / "runs" / "context23_class_plif_stream" / "best.pt"
print("热启动源 checkpoint:", FP32_CONTEXT_CHECKPOINT)

## 1. `HWClassAdaptiveContextSNN`：和 FP32 版本的结构差异

对照 `hw_model.py`：`encoder.0/encoder.1`（`Linear`+`LayerNorm`）被拆成
`enc_linear`(`QuantLinear`) + `enc_affine`(`HWAffine`)，`fc2`/`norm2` 同理。
`QuantLinear`/`QuantConv1d` 在前向时对权重做 `fake_quant_weight`（逐输出通道 INT4），
每次线性/仿射输出之后都要过一次 `fake_quant_activation`（Q8.8），衰减系数
`beta1`/`beta2` 也要经过 `fake_quant_decay`（k/256 网格）。这些函数定义在
`hw_ops.py`，就是教程 06 章讲的那套伪量化算子库的真实版本。

In [ ]:
hw_model = HWClassAdaptiveContextSNN(features=336).to(device)
print(hw_model)

## 2. 热启动：把 FP32 权重映射进 HW 模型

`remap_context_state` 把 `encoder.0.*` -> `enc_linear.*`、`encoder.1.*` -> `enc_affine.*`
改名，其余 key（LIF 衰减参数、`out` 层、`context_gamma_logit` 等）名字本来就一致，
直接对齐。加载后未被覆盖的参数（新引入的、FP32 版本没有的）保持默认初始化。

In [ ]:
fp32_state = torch.load(FP32_CONTEXT_CHECKPOINT, map_location=device, weights_only=False)["model"]
remapped = remap_context_state(fp32_state)
compatible = {k: v for k, v in remapped.items()
              if k in hw_model.state_dict() and hw_model.state_dict()[k].shape == v.shape}
missing, unexpected = hw_model.load_state_dict(compatible, strict=False)
print(f"loaded={len(compatible)}  missing={missing}  unexpected={unexpected}")

## 3. QAT 微调

超参数取自真实项目 `hw_context23_qat_v1` 的记录：`lr=2e-4, epochs=50, patience=15,
weight_power=0.2, label_smoothing=0.03`——比 FP32 训练的学习率低了一个量级，
因为热启动起点已经很接近好的解，只需要"适应"量化引入的噪声。

In [ ]:
@torch.no_grad()
def evaluate_hw_context(model, loader):
    model.eval()
    preds, targets = [], []
    for f, raw, y, subject in loader:
        output, _ = model(f.to(device), raw.to(device), subject.to(device))
        preds.extend(output.argmax(1).cpu().tolist())
        targets.extend(y.tolist())
    y_arr, p_arr = np.asarray(targets), np.asarray(preds)
    return {
        "accuracy": accuracy_score(y_arr, p_arr),
        "macro_f1": f1_score(y_arr, p_arr, average="macro"),
        "gesture_accuracy": float(np.mean(p_arr[y_arr != 0] == y_arr[y_arr != 0])),
    }


def train_qat_context(run_name, epochs, lr, patience, weight_power=0.2,
                       label_smoothing=0.03, batch_size=256, seed=42):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    sets = {
        split: EMGDataset(
            PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
            split == "train", context=23, continuous_context=True, stream_context=True,
        )
        for split in ("train", "val", "test")
    }
    loaders = {
        "train": DataLoader(sets["train"], batch_size, shuffle=True, num_workers=4, pin_memory=True),
        "val": DataLoader(sets["val"], batch_size * 2, num_workers=4, pin_memory=True),
        "test": DataLoader(sets["test"], batch_size * 2, num_workers=4, pin_memory=True),
    }
    optimizer = torch.optim.AdamW(hw_model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    counts = np.bincount(sets["train"].y, minlength=13)
    class_weights = torch.tensor((counts.sum() / (13 * counts)) ** weight_power,
                                  dtype=torch.float32, device=device)

    run_dir = NB_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_acc, stale = -1.0, 0

    for epoch in range(1, epochs + 1):
        hw_model.train()
        losses = []
        for f, raw, y, subject in loaders["train"]:
            f, raw, y, subject = f.to(device), raw.to(device), y.to(device), subject.to(device)
            optimizer.zero_grad(set_to_none=True)
            output, _ = hw_model(f, raw, subject)
            loss = nn.functional.cross_entropy(output, y, weight=class_weights,
                                                label_smoothing=label_smoothing)
            loss.backward()
            nn.utils.clip_grad_norm_(hw_model.parameters(), 2.0)
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()

        val_metrics = evaluate_hw_context(hw_model, loaders["val"])
        print(f"epoch {epoch:02d}  loss={np.mean(losses):.4f}  val_acc={val_metrics['accuracy']:.4f}")
        if val_metrics["accuracy"] > best_acc:
            best_acc, stale = val_metrics["accuracy"], 0
            torch.save({"model": hw_model.state_dict(), "epoch": epoch, "validation": val_metrics},
                       run_dir / "best.pt")
        else:
            stale += 1
            if stale >= patience:
                print("early stopping"); break

    ckpt = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    hw_model.load_state_dict(ckpt["model"])
    test_metrics = evaluate_hw_context(hw_model, loaders["test"])
    print(f"\n=== {run_name} 最终结果（第 {ckpt['epoch']} 轮）===")
    print(f"val:  accuracy={ckpt['validation']['accuracy']:.4f}")
    print(f"test: accuracy={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}")
    return run_dir / "best.pt", test_metrics

qat_context_checkpoint, qat_context_test_metrics = train_qat_context(
    run_name="hw_context23_qat_nb", epochs=50, lr=2e-4, patience=15,
)
print("\n参考值(真实项目 hw_context23_qat_v1_affinefix): test accuracy≈0.8891")

## 4. FP32 vs QAT 精度对照

真实项目的经验是：只要算子替换得当、走完整的 QAT 微调，量化几乎不掉点
（有时甚至因为量化起到轻微正则化作用而略微提升）。

In [ ]:
try:
    fp32_row = stage2_test_metrics  # 来自 notebook 01,如果在同一个 kernel 里连续跑过
except NameError:
    fp32_row = {"accuracy": 0.8853, "macro_f1": 0.7822, "gesture_accuracy": 0.7376}
    print("(未在本 session 里运行过 notebook 01,用 RESULTS.md 记录的参考值代替)\n")

print(f"{'':12s}{'accuracy':>10s}{'macro_f1':>10s}{'gesture_acc':>12s}")
print(f"{'FP32':12s}{fp32_row['accuracy']:10.4f}{fp32_row['macro_f1']:10.4f}{fp32_row['gesture_accuracy']:12.4f}")
print(f"{'HW-QAT':12s}{qat_context_test_metrics['accuracy']:10.4f}"
      f"{qat_context_test_metrics['macro_f1']:10.4f}{qat_context_test_metrics['gesture_accuracy']:12.4f}")

## 5. 一个真实的量化 bug 案例(教程 06.10 节)

早期版本的 `HWAffine`（`hw_ops.py`）曾经有一个 bug：错误地对一维的仿射权重做了
"逐输出通道"缩放，而缩放的归约维度大小恰好是 1，导致 `scale = |w_i|/limit`，
量化后精确无损重建原始浮点值——**看起来在量化,实际上什么都没量化**。
这个 bug 是通过和下面第 6 节的 numpy 交叉验证发现的。现在 `hw_ops.py` 里的
`fake_quant_tensor` 已经是修复后的版本(整个张量共享一个 scale)。这里做一个诊断:
检查 `enc_affine`/`norm2` 权重实际用到的量化编码值范围,确认量化确实"生效"了
(编码值集中在某个有限范围,不是每个都精确等于 ±limit 的边界值)。

In [ ]:
from hw_ops import fake_quant_tensor

with torch.no_grad():
    for name, module in (("enc_affine", hw_model.enc_affine), ("norm2", hw_model.norm2)):
        weight = module.weight.detach()
        limit = 2 ** (module.bits - 1) - 1
        amax = weight.abs().amax().clamp_min(1e-8)
        scale = amax / limit
        codes = torch.round(weight / scale).clamp(-limit, limit)
        print(f"{name}: bits={module.bits}  scale={float(scale):.5f}  "
              f"code range used=[{int(codes.min())}, {int(codes.max())}]  "
              f"distinct codes={codes.unique().numel()} / possible={2*limit+1}")

`distinct codes` 明显小于 `possible`、编码值也不是每个都精确等于 `±limit`，
说明量化是真实生效的（如果是那个 bug，编码值会永远等于 `±limit`）。

## 6. 导出定点权重 + numpy 参考实现交叉验证

对应教程 07 章：把伪量化的浮点权重变成真正的 `int8 codes + scale`，写一份不依赖
torch 的 numpy 推理，验证它和 QAT 训练时的 torch 前向在同一批验证样本上给出
一致的预测。这里复用真实项目的 `export_hw_fixed.py::export_context()` 逻辑。

In [ ]:
def export_context_fixed(state: dict, weight_bits: int, out_path: Path) -> None:
    """精简版 export_hw_fixed.py::export_context(),逻辑完全一致。"""
    arrays = {}

    def add_linear(prefix, weight_key, bias_key, bits):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 2 ** (bits - 1) - 1
        amax = np.maximum(np.abs(weight).max(axis=tuple(range(1, weight.ndim)), keepdims=True), 1e-8)
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = scale.reshape(-1).astype(np.float32)
        arrays[f"{prefix}_bias"] = bias

    def add_affine(prefix, weight_key, bias_key):
        weight = state[weight_key].detach().cpu().numpy()
        bias = state[bias_key].detach().cpu().numpy().astype(np.float32)
        limit = 127
        amax = float(np.maximum(np.abs(weight).max(), 1e-8))
        scale = amax / limit
        codes = np.clip(np.round(weight / scale), -limit, limit).astype(np.int8)
        arrays[f"{prefix}_codes"] = codes
        arrays[f"{prefix}_scale"] = np.float32(scale)
        arrays[f"{prefix}_bias"] = bias

    add_linear("enc_linear", "enc_linear.weight", "enc_linear.bias", weight_bits)
    add_affine("enc_affine", "enc_affine.weight", "enc_affine.bias")
    add_linear("fc2", "fc2.weight", "fc2.bias", weight_bits)
    add_affine("norm2", "norm2.weight", "norm2.bias")
    add_linear("out", "out.weight", "out.bias", weight_bits)

    def quantize_decay(beta, frac_bits=8):
        scale = 2.0 ** (-frac_bits)
        return float(np.clip(np.round(beta / scale), 1, 2 ** frac_bits) * scale)

    beta1 = torch.sigmoid(state["beta1"] + 0.5 * torch.tanh(state["beta1_offset"])).detach().cpu().numpy()
    beta2 = torch.sigmoid(state["beta2"] + 0.5 * torch.tanh(state["beta2_offset"])).detach().cpu().numpy()
    arrays["beta1"] = np.array([quantize_decay(v) for v in beta1], dtype=np.float32)
    arrays["beta2"] = np.array([quantize_decay(v) for v in beta2], dtype=np.float32)
    arrays["context_gamma"] = torch.sigmoid(state["context_gamma_logit"]).detach().cpu().numpy().astype(np.float32)
    arrays["act_frac_bits"], arrays["act_int_bits"] = np.int32(8), np.int32(8)
    arrays["substeps"], arrays["windows"] = np.int32(3), np.int32(23)

    np.savez(out_path, **arrays)
    total_bytes = sum(a.nbytes for a in arrays.values() if hasattr(a, "nbytes"))
    print(f"wrote {out_path} ({total_bytes/1024:.1f} KiB)")

final_state = hw_model.state_dict()
export_context_fixed(final_state, weight_bits=4, out_path=WEIGHTS_DIR / "hw_context_fixed_nb.npz")

In [ ]:
# 交叉验证:torch QAT 模型 vs 纯 numpy 定点参考实现
numpy_model = HWFixedContext(WEIGHTS_DIR / "hw_context_fixed_nb.npz")

val_dataset = EMGDataset(
    PROJECT_ROOT / "data" / "val.npz", PROJECT_ROOT / "data" / "normalization.npz",
    False, context=23, continuous_context=True, stream_context=True,
)
sample_idx = np.random.RandomState(0).choice(len(val_dataset), size=256, replace=False)
features_batch = torch.stack([val_dataset[i][0] for i in sample_idx]).to(device)

hw_model.eval()
with torch.no_grad():
    torch_logits, _ = hw_model(features_batch, None, None)
torch_logits = torch_logits.cpu().numpy()

numpy_logits, numpy_argmax = numpy_model.infer(features_batch.cpu().numpy())

match_rate = (torch_logits.argmax(1) == numpy_argmax).mean()
max_diff = np.abs(torch_logits - numpy_logits).max()
print(f"argmax 一致率: {match_rate:.4f}  (预期 1.0)")
print(f"最大 logit 差异: {max_diff:.6f}  (预期是浮点求和顺序造成的小残差,不是 0)")

argmax 一致率应该是 100%——这证明导出的 `int8 codes + scale` 定点规格和 QAT
训练时 torch 模型实际学到的行为是位对位一致的，这份 `.npz` 才有资格被交给
硬件工程师当作"规格书"使用。

## 下一步

打开 [03_hybrid_training.ipynb](03_hybrid_training.ipynb)，用同样的思路处理
Hybrid 分支——它比 Context 多几种算子（GELU、BatchNorm、Jaccard 除法），
量化时要处理的替换清单更长。